# Lab 19-03 · Logic Families: Stacks, Ratios, Pass Gates and Dynamic Nodes

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import logic as lg

## Part 1 · Fan-in

*Predict* the NAND-N fall delay for N = 1 to 12 (R = 10 kOhm, 0.5 fF per unit width, 10 fF load) and where the internal-node term overtakes the output term.

In [2]:
R, c, cext = 10e3, 0.5e-15, 10e-15
for n in range(1, 13):
    tau = lg.nand_stack_elmore(n, R, c, cext)
    quad = R * c * n * (n - 1) / 2
    print(f"N = {n:2d}: tau = {tau * 1e12:6.1f} ps, delay {lg.LN2 * tau * 1e12:6.1f} ps, internal-node share {quad / tau:5.1%}")
n_eq = (7 + math.sqrt(49 + 160)) / 2
print(f"internal-node term equals output term at N = {n_eq:.2f}")

N =  1: tau =  115.0 ps, delay   79.7 ps, internal-node share  0.0%
N =  2: tau =  135.0 ps, delay   93.6 ps, internal-node share  3.7%
N =  3: tau =  160.0 ps, delay  110.9 ps, internal-node share  9.4%
N =  4: tau =  190.0 ps, delay  131.7 ps, internal-node share 15.8%
N =  5: tau =  225.0 ps, delay  156.0 ps, internal-node share 22.2%
N =  6: tau =  265.0 ps, delay  183.7 ps, internal-node share 28.3%
N =  7: tau =  310.0 ps, delay  214.9 ps, internal-node share 33.9%
N =  8: tau =  360.0 ps, delay  249.5 ps, internal-node share 38.9%
N =  9: tau =  415.0 ps, delay  287.7 ps, internal-node share 43.4%
N = 10: tau =  475.0 ps, delay  329.2 ps, internal-node share 47.4%
N = 11: tau =  540.0 ps, delay  374.3 ps, internal-node share 50.9%
N = 12: tau =  610.0 ps, delay  422.8 ps, internal-node share 54.1%
internal-node term equals output term at N = 10.73


## Part 2 · Elmore against exact

*Predict* how the ratio of the exact 50% delay to 0.69 tau_E changes with the number of ladder sections (fixed total R and C).

In [3]:
for n in (1, 2, 4, 12, 30):
    r = [1.0 / n] * n; cc = [1.0 / n] * n
    exact = lg.ladder_step_50(r, cc)
    est = lg.LN2 * lg.elmore_ladder(r, cc)
    print(f"{n:2d} sections: exact {exact:.4f} RC, estimate {est:.4f} RC, ratio {exact / est:.4f}")
print("12 TGs of 5 kOhm and 2 fF:", round(lg.ladder_step_50([5e3] * 12, [2e-15] * 12) * 1e12, 1), "ps (ngspice: tg_ladder.cir)")

 1 sections: exact 0.6931 RC, estimate 0.6931 RC, ratio 1.0000
 2 sections: exact 0.5562 RC, estimate 0.5199 RC, ratio 1.0700
 4 sections: exact 0.4709 RC, estimate 0.4332 RC, ratio 1.0871
12 sections: exact 0.4101 RC, estimate 0.3755 RC, ratio 1.0921
30 sections: exact 0.3913 RC, estimate 0.3581 RC, ratio 1.0927
12 TGs of 5 kOhm and 2 fF: 590.5 ps (ngspice: tg_ladder.cir)


## Part 3 · Pseudo-NMOS

*Predict* V_OL and the static current of Example 19-03.3, and how V_OL scales with kn/kp.

In [4]:
print(f"V_OL = {lg.pseudo_nmos_vol(200e-6, 50e-6, 1.2, 0.35, 0.35):.4f} V, I = {lg.pseudo_nmos_static_current(50e-6, 1.2, 0.35) * 1e6:.2f} uA")
ratios = np.linspace(2, 12, 51)
vols = [lg.pseudo_nmos_vol(200e-6, 200e-6 / r, 1.2, 0.35, 0.35) for r in ratios]
plt.plot(ratios, vols); plt.xlabel("kn / kp"); plt.ylabel("V_OL (V)"); plt.show()
print("ratio for 0.1 V:", round(lg.pseudo_nmos_ratio_for_vol(0.1, 1.2, 0.35, 0.35), 3))

V_OL = 0.1139 V, I = 18.06 uA
ratio for 0.1 V: 4.516


## Part 4 · Pass transistor

*Predict* the passed level for gamma_B = 0, 0.4 and 0.6, and the current of the inverter that follows.

In [5]:
for g in (0.0, 0.4, 0.6):
    vh = lg.nmos_pass_high(1.2, 0.35, g, 0.8)
    vo, i = lg.inverter_static_current(vh, 1.2, 0.35, 0.35, 200e-6, 200e-6)
    print(f"gamma_B = {g}: passed level {vh:.4f} V; next inverter output {vo * 1e3:.1f} mV, current {i * 1e6:.3f} uA")

gamma_B = 0.0: passed level 0.8500 V; next inverter output 0.0 mV, current 0.000 uA
gamma_B = 0.4: passed level 0.7154 V; next inverter output 25.7 mV, current 1.813 uA
gamma_B = 0.6: passed level 0.6613 V; next inverter output 63.7 mV, current 3.559 uA


## Part 5 · Buffered TG chains

*Predict* the best segment length for buffer delays of 10, 30 and 100 ps (5 kOhm, 2 fF per section).

In [6]:
for tb in (10e-12, 30e-12, 100e-12):
    m_opt = lg.tg_optimal_m(5e3, 2e-15, tb)
    delays = {m: lg.tg_chain_buffered(12, m, 5e3, 2e-15, tb) * 1e12 for m in (1, 2, 3, 4, 6, 12)}
    best = min(delays, key=delays.get)
    print(f"t_buf = {tb * 1e12:.0f} ps: m_opt = {m_opt:.2f}; best divisor of 12: m = {best} ({delays[best]:.1f} ps)")

t_buf = 10 ps: m_opt = 1.70; best divisor of 12: m = 2 (174.8 ps)
t_buf = 30 ps: m_opt = 2.94; best divisor of 12: m = 3 (256.4 ps)
t_buf = 100 ps: m_opt = 5.37; best divisor of 12: m = 6 (391.1 ps)


## Part 6 · Charge sharing

*Predict* the output after charge sharing for C_a from 0 to 20 fF (C_L = 10 fF), and where it crosses 0.6 V.

In [7]:
ca = np.linspace(0, 20e-15, 201)
vout = [lg.charge_sharing(1.2, 0.35, 10e-15, x) for x in ca]
plt.plot(ca * 1e15, vout); plt.axhline(0.6, color="r"); plt.xlabel("C_a (fF)"); plt.ylabel("output (V)"); plt.show()
print("3 fF:", round(lg.charge_sharing(1.2, 0.35, 10e-15, 3e-15), 4), "V; 12 fF:", round(lg.charge_sharing(1.2, 0.35, 10e-15, 12e-15), 4), "V")
print("hold time for 1 nA and 0.1 V:", lg.hold_time(10e-15, 0.1, 1e-9) * 1e6, "us")

3 fF: 0.945 V; 12 fF: 0.5455 V
hold time for 1 nA and 0.1 V: 1.0 us


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).